In [4]:
# --- Notebook cell: compute multihot mechanisms for label==1, then print representative examples per flag ---
# Adds a "reasonable minimum length" filter so you don't pick tiny / trivial strings.

import os
import importlib.util
import numpy as np
import pandas as pd
from typing import List, Optional
from IPython.display import display

# -----------------------
# CONFIG (edit these)
# -----------------------
PARQUET_PATH = "text_test.parquet"              # parquet with: fraudulent_name, real_name, label
MECH_SCRIPT_PATH = "mechanism_multihot_audit.py"  # defines FLAG_KEYS + mechanism_flags_df + combo_id + combo_label

FRAUD_COL = "fraudulent_name"
REAL_COL  = "real_name"
LABEL_COL = "label"

K_PER_FLAG = 5                  # set to 1 for exactly one example per flag
ONLY_POSITIVES = True           # print examples from label==1 only
PREFER_PURE = True              # prefer rows where this flag is on + few other flags are on
DIVERSIFY_BY_REAL = True        # avoid repeating the same real_name across examples
SEED = 0

# "reasonable min length" filters (tune these)
MIN_LEN_FRAUD = 6               # require len(fraudulent_name) >= this (0 disables)
MIN_LEN_REAL  = 6               # require len(real_name) >= this (0 disables)
MIN_LEN_SUM   = 0               # require len(fraud)+len(real) >= this (0 disables)

# if a flag has *no* examples meeting min-length, fall back to allowing shorter ones (so you still get something)
FALLBACK_ALLOW_SHORT = True

DISPLAY_COLS_BASE = [LABEL_COL, FRAUD_COL, REAL_COL]  # extra cols auto-added if present


# -----------------------
# Helpers
# -----------------------
def _ensure_cols(df: pd.DataFrame, cols: List[str]) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in parquet: {missing}")


def load_module_from_path(py_path: str):
    if not os.path.exists(py_path):
        raise FileNotFoundError(f"Mechanism script not found at: {py_path}")
    spec = importlib.util.spec_from_file_location("mech_module", py_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not import module from: {py_path}")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)  # type: ignore[attr-defined]
    return mod


def compute_flags_for_label1(df: pd.DataFrame, mech_mod) -> pd.DataFrame:
    """
    Uses your existing:
      - mech_mod.FLAG_KEYS
      - mech_mod.mechanism_flags_df(fraud_series, real_series)
      - mech_mod.combo_id(flags_row)
      - mech_mod.combo_label(flags_row)

    Computes flags ONLY for label==1; others remain all-zero.
    """
    FLAG_KEYS = list(mech_mod.FLAG_KEYS)

    if not hasattr(mech_mod, "mechanism_flags_df"):
        raise AttributeError("Your mechanism script must define mechanism_flags_df(fraud_series, real_series).")
    if not hasattr(mech_mod, "combo_id") or not hasattr(mech_mod, "combo_label"):
        raise AttributeError("Your mechanism script must define combo_id(flags_row) and combo_label(flags_row).")

    flags = pd.DataFrame(0, index=df.index, columns=FLAG_KEYS, dtype=int)

    mask_pos = df[LABEL_COL] == 1
    if mask_pos.any():
        flags_pos = mech_mod.mechanism_flags_df(
            df.loc[mask_pos, FRAUD_COL].astype(str),
            df.loc[mask_pos, REAL_COL].astype(str),
        )
        flags_pos = flags_pos.reindex(columns=FLAG_KEYS).fillna(0).astype(int)
        flags.loc[mask_pos, :] = flags_pos.values

    out = pd.concat([df.reset_index(drop=True), flags.reset_index(drop=True)], axis=1)

    out["mech_n_flags"] = flags.sum(axis=1).astype(int).reset_index(drop=True)
    out["mech_flags_str"] = flags.apply(
        lambda r: "+".join([k for k in FLAG_KEYS if int(r[k]) == 1]) if int(r.sum()) > 0 else "NONE",
        axis=1,
    ).reset_index(drop=True)

    out["mech_combo_id"] = [int(mech_mod.combo_id(r)) for _, r in flags.iterrows()]
    out["mech_combo"] = [str(mech_mod.combo_label(r)) for _, r in flags.iterrows()]

    return out


def _apply_minlen_filter(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies MIN_LEN_* filters using FRAUD_COL/REAL_COL.
    Assumes df already contains those columns.
    """
    out = df.copy()

    lf = out[FRAUD_COL].astype(str).str.len().fillna(0).astype(int)
    lr = out[REAL_COL].astype(str).str.len().fillna(0).astype(int)
    ls = (lf + lr).astype(int)

    mask = pd.Series(True, index=out.index)
    if MIN_LEN_FRAUD and MIN_LEN_FRAUD > 0:
        mask &= (lf >= int(MIN_LEN_FRAUD))
    if MIN_LEN_REAL and MIN_LEN_REAL > 0:
        mask &= (lr >= int(MIN_LEN_REAL))
    if MIN_LEN_SUM and MIN_LEN_SUM > 0:
        mask &= (ls >= int(MIN_LEN_SUM))

    return out.loc[mask].copy()


def pick_representative_examples(
    df: pd.DataFrame,
    flag_keys: List[str],
    flag: str,
    k: int,
    prefer_pure: bool,
    diversify_by_real: bool,
    seed: int,
    only_positives: bool,
) -> pd.DataFrame:
    if flag not in df.columns:
        return df.head(0).copy()

    base = df[df[flag] == 1].copy()
    if only_positives and LABEL_COL in base.columns:
        base = base[base[LABEL_COL] == 1].copy()
    if len(base) == 0:
        return base

    # apply min-length filter; optionally fall back
    sub = _apply_minlen_filter(base)
    if len(sub) == 0 and FALLBACK_ALLOW_SHORT:
        sub = base.copy()

    rng = np.random.default_rng(seed)
    sub["_tie"] = rng.random(len(sub))

    other_flags = [c for c in flag_keys if c in sub.columns and c != flag]
    sub["_n_other_flags"] = sub[other_flags].sum(axis=1).astype(int) if other_flags else 0

    sub["_len_fraud"] = sub[FRAUD_COL].astype(str).str.len().fillna(0).astype(int)
    sub["_len_real"]  = sub[REAL_COL].astype(str).str.len().fillna(0).astype(int)
    sub["_len_sum"]   = (sub["_len_fraud"] + sub["_len_real"]).astype(int)

    sort_cols = (["_n_other_flags"] if prefer_pure else []) + ["_len_sum", "_tie"]
    sub = sub.sort_values(sort_cols, ascending=True)

    if diversify_by_real:
        picked = []
        seen_real = set()
        for _, row in sub.iterrows():
            r = str(row[REAL_COL])
            if r in seen_real:
                continue
            picked.append(row)
            seen_real.add(r)
            if len(picked) >= k:
                break
        out = pd.DataFrame(picked) if picked else sub.head(0).copy()
    else:
        out = sub.head(k)

    return out.drop(columns=[c for c in out.columns if c.startswith("_")], errors="ignore")


def show_examples_for_all_flags(
    df: pd.DataFrame,
    flag_keys: List[str],
    k_per_flag: int,
    prefer_pure: bool,
    diversify_by_real: bool,
    seed: int,
    only_positives: bool,
    display_cols_base: Optional[List[str]] = None,
) -> None:
    if display_cols_base is None:
        display_cols_base = DISPLAY_COLS_BASE

    # counts (match the min-length filter; if fallback is enabled, counts may under-report for some flags)
    counts = []
    for f in flag_keys:
        if f not in df.columns:
            counts.append((f, 0, False))
            continue
        sub = df[df[f] == 1]
        if only_positives and LABEL_COL in df.columns:
            sub = sub[sub[LABEL_COL] == 1]
        sub = _apply_minlen_filter(sub)
        counts.append((f, int(len(sub)), True))

    counts_df = pd.DataFrame(counts, columns=["flag", "n_rows_minlen", "present_in_df"])
    counts_df = counts_df.sort_values(["present_in_df", "n_rows_minlen", "flag"], ascending=[False, False, True])

    print("Flag counts (label filter + min-length filter):")
    display(counts_df)

    for flag, n_rows, present in counts_df.itertuples(index=False):
        if not present:
            continue

        print("\n" + "=" * 100)
        print(f"{flag}  (n_minlen={n_rows})")

        ex = pick_representative_examples(
            df=df,
            flag_keys=flag_keys,
            flag=flag,
            k=k_per_flag,
            prefer_pure=prefer_pure,
            diversify_by_real=diversify_by_real,
            seed=seed,
            only_positives=only_positives,
        )

        if len(ex) == 0:
            print("(no rows)")
            continue

        cols = []
        cols += [c for c in display_cols_base if c in ex.columns]
        for c in ["mech_combo", "mech_combo_id", "mech_n_flags", "mech_flags_str"]:
            if c in ex.columns and c not in cols:
                cols.append(c)
        if flag in ex.columns and flag not in cols:
            cols.append(flag)

        display(ex[cols].reset_index(drop=True))


# -----------------------
# RUN
# -----------------------
df0 = pd.read_parquet(PARQUET_PATH)
_ensure_cols(df0, [FRAUD_COL, REAL_COL, LABEL_COL])

mech_mod = load_module_from_path(MECH_SCRIPT_PATH)
FLAG_KEYS = list(mech_mod.FLAG_KEYS)

df = compute_flags_for_label1(df0, mech_mod)

show_examples_for_all_flags(
    df=df,
    flag_keys=FLAG_KEYS,
    k_per_flag=K_PER_FLAG,
    prefer_pure=PREFER_PURE,
    diversify_by_real=DIVERSIFY_BY_REAL,
    seed=SEED,
    only_positives=ONLY_POSITIVES,
    display_cols_base=DISPLAY_COLS_BASE,
)

# Usage:
# - Adjust MIN_LEN_FRAUD / MIN_LEN_REAL (and optionally MIN_LEN_SUM)
# - Set K_PER_FLAG = 1 for exactly one example per mechanism

Flag counts (label filter + min-length filter):


,flag,n_rows_minlen,present_in_df
13,substitution,113127,True
3,non_ascii,87231,True
11,insertion,53296,True
5,unicode_homoglyph,24982,True
4,unicode_marks_only,24982,True
12,deletion,16358,True
8,hyphen_change,8934,True
9,digit_change,8472,True
19,affix,5906,True
17,separator_change,3217,True



substitution  (n_minlen=113127)


,label,fraudulent_name,real_name,mech_combo,mech_combo_id,mech_n_flags,mech_flags_str,substitution
0,1.0,hefree,hdfree,substitution,8192,1,substitution,1
1,1.0,stnmaz,sunmar,substitution,8192,1,substitution,1
2,1.0,sezuki,suzuki,substitution,8192,1,substitution,1
3,1.0,tehkku,telkku,substitution,8192,1,substitution,1
4,1.0,iuxlij,muxlij,substitution,8192,1,substitution,1



non_ascii  (n_minlen=87231)


...


multichar_confusable  (n_minlen=93)


,label,fraudulent_name,real_name,mech_combo,mech_combo_id,mech_n_flags,mech_flags_str,multichar_confusable
0,1.0,clata18,data18,substitution+visual_confusable+multichar_confu...,6299648,3,substitution+visual_confusable+multichar_confu...,1
1,1.0,rnailce,mailce,substitution+visual_confusable+multichar_confu...,6299648,3,substitution+visual_confusable+multichar_confu...,1
2,1.0,zrnovie,zmovie,substitution+visual_confusable+multichar_confu...,6299648,3,substitution+visual_confusable+multichar_confu...,1
3,1.0,iclefix,idefix,substitution+visual_confusable+multichar_confu...,6299648,3,substitution+visual_confusable+multichar_confu...,1
4,1.0,asrnnet,asmnet,substitution+visual_confusable+multichar_confu...,6299648,3,substitution+visual_confusable+multichar_confu...,1



numeric_affix  (n_minlen=26)


,label,fraudulent_name,real_name,mech_combo,mech_combo_id,mech_n_flags,mech_flags_str,numeric_affix
0,1.0,7learn5,7learn,insertion+affix+numeric_affix,1574912,3,insertion+affix+numeric_affix,1
1,1.0,0times,10times,deletion+affix+numeric_affix,1576960,3,deletion+affix+numeric_affix,1
2,1.0,flv2mp,flv2mp3,deletion+affix+numeric_affix,1576960,3,deletion+affix+numeric_affix,1
3,1.0,judou12,judou123,deletion+affix+numeric_affix,1576960,3,deletion+affix+numeric_affix,1
4,1.0,defence2,defence24,deletion+affix+numeric_affix,1576960,3,deletion+affix+numeric_affix,1



case_change_only  (n_minlen=0)
(no rows)

extension  (n_minlen=0)
(no rows)

whitespace_change  (n_minlen=0)
(no rows)

zero_width_or_format  (n_minlen=0)
(no rows)
